In [1]:
#cell 1
!apt-get install -y -qq ffmpeg espeak-ng libsndfile1 libopus-dev libvpx-dev
print('✅ System packages installed')

✅ System packages installed


In [2]:
#cell 2
!pip install -q \
    faster-whisper \
    "kokoro>=0.9.4" \
    soundfile \
    groq \
    aiortc \
    aiohttp \
    av \
    fastapi \
    uvicorn \
    pyngrok \
    numpy

print('✅ Python packages installed')

✅ Python packages installed


In [3]:
#cell 3
from groq import Groq

GROQ_API_KEY = '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'   # ← your key from console.groq.com
NGROK_TOKEN  = '3DIZw3g5LOd4Hg0S9KMilmPuJMW_4xSdBvtY4eaetzDNzRdEJ'        # ← your key from dashboard.ngrok.com
MODEL        = 'llama-3.1-8b-instant'
METERED_USERNAME   = 'bcd0b353237c5c18d7143640'            # ← paste here
METERED_CREDENTIAL = 'YgVR8KLicmcHdfJ8'

client = Groq(api_key=GROQ_API_KEY)

print('🔍 Testing Groq...')
try:
    _t = client.chat.completions.create(
        model=MODEL,
        messages=[{'role':'user','content':'Reply with exactly: OK'}],
        max_tokens=5
    )
    print(f'✅ Groq OK → {_t.choices[0].message.content.strip()}')
except Exception as e:
    print(f'❌ Groq FAILED: {e}'); raise

🔍 Testing Groq...
✅ Groq OK → OK


In [4]:
#cell 4
from faster_whisper import WhisperModel
from kokoro import KPipeline
import torch, numpy as np, soundfile as sf

print(f'GPU: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'  → {torch.cuda.get_device_name(0)}')

print('⏳ Loading Whisper small...')
stt = WhisperModel(
    'medium',
    device      = 'cuda' if torch.cuda.is_available() else 'cpu',
    compute_type= 'float16' if torch.cuda.is_available() else 'int8'
)
print('✅ Whisper ready!')

print('⏳ Loading Kokoro TTS...')
tts_model = KPipeline(lang_code='a')
_ = list(tts_model('Hello!', voice='af_sky', speed=1.0))   # warm-up
print('✅ Kokoro ready!')

GPU: True
  → Tesla T4
⏳ Loading Whisper small...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


✅ Whisper ready!
⏳ Loading Kokoro TTS...


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/rnn.py:990: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  super().__init__("LSTM", *args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


✅ Kokoro ready!


In [5]:
import time, numpy as np, soundfile as sf
from collections import Counter

# ── Emotion config ────────────────────────────────────────────────────────────
EMOTION_CONFIG = {
    'happy':   {'voice': 'af_heart', 'speed': 1.15},
    'sad':     {'voice': 'af_sky',   'speed': 0.82},
    'angry':   {'voice': 'am_adam',  'speed': 1.05},
    'excited': {'voice': 'af_heart', 'speed': 1.25},
    'calm':    {'voice': 'af_sky',   'speed': 0.95},
    'neutral': {'voice': 'af_sky',   'speed': 1.00},
}
EMOTION_KEYWORDS = {
    'angry':   ['angry','furious','frustrated','unacceptable','terrible','worst'],
    'sad':     ['sad','upset','disappointed','depressed','unhappy','horrible'],
    'happy':   ['happy','amazing','love','wonderful','fantastic','thank you'],
    'excited': ['excited','awesome','thrilled','incredible'],
}

def detect_emotion(text):
    t = text.lower()
    for emotion, kws in EMOTION_KEYWORDS.items():
        if any(k in t for k in kws):
            mapped = {'angry':'calm','sad':'sad','happy':'happy','excited':'excited'}.get(emotion,'neutral')
            print(f'   Emotion: {emotion} → {mapped}')
            return mapped
    return 'neutral'

# ── Hallucination word list ───────────────────────────────────────────────────
HALLUCINATION_WORDS = {
    'transcribe','exactly','incomplete','sentences','including',
    'translated','subtitles','captions','transcript',
}

# ── Transcribe ────────────────────────────────────────────────────────────────
def transcribe(audio_array, sr):
    if sr != 16000:
        n = int(len(audio_array) * 16000 / sr)
        audio_array = np.interp(
            np.linspace(0, len(audio_array)-1, n),
            np.arange(len(audio_array)), audio_array
        ).astype(np.float32)

    duration = len(audio_array) / 16000
    if duration < 0.5:
        print(f'   Skip — too short ({duration:.1f}s)')
        return ''

    peak = float(np.abs(audio_array).max())
    rms  = float(np.sqrt(np.mean(audio_array**2)))
    print(f'   Audio stats — peak={peak:.4f} rms={rms:.5f} dur={duration:.2f}s')

    if peak < 0.001 or rms < 0.0008:
        print(f'   Skip — too quiet')
        return ''

    # Normalize
    if peak > 0.0:
        audio_array = audio_array * (0.7 / peak)
    audio_array = np.clip(audio_array, -1.0, 1.0)

    # ── Trim leading/trailing silence before sending to Whisper ──────────────
    # Long silent gaps cause Whisper to hallucinate wrong words
    TRIM_RMS_THRESHOLD = 0.015
    FRAME_SIZE         = 1600   # 0.1s at 16kHz
    frames = [audio_array[i:i+FRAME_SIZE]
              for i in range(0, len(audio_array)-FRAME_SIZE, FRAME_SIZE)]
    speech_frames_idx = [
        i for i, f in enumerate(frames)
        if float(np.sqrt(np.mean(f**2))) > TRIM_RMS_THRESHOLD
    ]

    if not speech_frames_idx:
        print('   Skip — no speech frames after trimming')
        return ''

    # Keep from first to last speech frame + 0.3s padding each side
    PAD   = 3   # frames = 0.3s
    start = max(0, speech_frames_idx[0]  - PAD) * FRAME_SIZE
    end   = min(len(audio_array),
                (speech_frames_idx[-1] + 1 + PAD) * FRAME_SIZE)
    audio_array = audio_array[start:end]

    trimmed_dur = len(audio_array) / 16000
    print(f'   Trimmed to {trimmed_dur:.2f}s of actual speech')

    if trimmed_dur < 0.4:
        print('   Skip — trimmed too short')
        return ''

    sf.write('/tmp/webrtc_user.wav', audio_array, 16000)

    segments, info = stt.transcribe(
        '/tmp/webrtc_user.wav',
        beam_size                   = 5,
        language                    = 'en',
        task                        = 'transcribe',
        vad_filter                  = True,
        vad_parameters              = dict(
            min_silence_duration_ms = 400,
            speech_pad_ms           = 300,
            threshold               = 0.30,
        ),
        condition_on_previous_text  = False,
        temperature                 = 0.0,
        no_speech_threshold         = 0.55,
        compression_ratio_threshold = 2.3,
        log_prob_threshold          = -0.9,
        word_timestamps             = False,
        initial_prompt = (
            'This is a casual voice conversation. The user may ask questions, '
            'do math like adding apples, or discuss everyday topics. '
            'Transcribe exactly what is said.'
        ),
    )

    text  = ' '.join(s.text for s in segments).strip()
    words = text.split()

    if not words:
        print('   Whisper returned empty')
        return ''

    lower = [w.lower().strip('.,!?"\'') for w in words]
    if sum(1 for w in lower if w in HALLUCINATION_WORDS) >= 2:
        print(f'   Hallucination suppressed: "{text}"')
        return ''

    if len(words) > 3:
        top_word, top_count = Counter(lower).most_common(1)[0]
        if top_count / len(words) > 0.5:
            print(f'   Repetition suppressed: "{top_word}" x{top_count}')
            return ''

    FILLERS = {'um','uh','hmm','hm','ah','oh','okay','ok','yeah','mhm','mm'}
    if len(words) == 1 and lower[0] in FILLERS:
        return ''

    print(f'   [{info.language}] "{text}"')
    return text


# ── LLM ───────────────────────────────────────────────────────────────────────
def ask_ai(user_text, history):
    if not user_text or len(user_text.strip()) < 3:
        return "I didn't catch that — could you say it again?"

    messages = [{'role':'system','content':(
        'You are a friendly, empathetic AI voice assistant on a phone call. '
        'Keep every reply to 1-2 short conversational sentences. '
        'Do NOT use bullet points, lists, or markdown. '
        'Speak naturally as if on a real phone call.'
    )}]
    for u, a in history[-6:]:
        messages.append({'role':'user',      'content': u})
        messages.append({'role':'assistant', 'content': a})
    messages.append({'role':'user','content': user_text})

    try:
        resp  = client.chat.completions.create(
            model=MODEL, messages=messages,
            max_tokens=80, temperature=0.75, top_p=0.9
        )
        reply = resp.choices[0].message.content.strip()
        print(f'   AI: "{reply}"')
        return reply or "I'm not sure — could you say that again?"
    except Exception as ex:
        print(f'   Groq error: {ex}')
        return 'Sorry, I had a brief hiccup — please go ahead.'


# ── TTS ───────────────────────────────────────────────────────────────────────
def speak(text, emotion='neutral'):
    cfg = EMOTION_CONFIG.get(emotion, EMOTION_CONFIG['neutral'])
    try:
        chunks = [a for _, _, a in tts_model(text, voice=cfg['voice'], speed=cfg['speed'])]
        if not chunks:
            raise ValueError('TTS empty')
        audio = np.concatenate(chunks).astype(np.float32)
        print(f'   TTS generated {len(audio)/24000:.2f}s audio, max={float(np.abs(audio).max()):.3f}')
        return (24000, audio)
    except Exception as ex:
        print(f'   TTS error: {ex}')
        return (24000, np.zeros(12000, dtype=np.float32))

print('✅ Helper functions ready!')

✅ Helper functions ready!


In [6]:
import asyncio, fractions, uuid, numpy as np
from aiortc import (RTCPeerConnection, RTCSessionDescription,
                    RTCConfiguration, RTCIceServer, MediaStreamTrack)
from fastapi import FastAPI, Request
from fastapi.responses import HTMLResponse, JSONResponse
import av

pcs = set()

# ── VAD tuning ────────────────────────────────────────────────────────────────
ENERGY_THRESHOLD      = 0.015
MIN_SPEECH_FRAMES     = 12
SILENCE_THRESHOLD_SEC = 2.0
MIN_SPEECH_SEC        = 1.2
MAX_BUFFER_SEC        = 25.0
PRE_SPEECH_KEEP_SEC   = 0.3

# ── ICE / TURN config ─────────────────────────────────────────────────────────
ICE_CONFIG = RTCConfiguration(iceServers=[
    RTCIceServer(urls=['stun:stun.metered.ca:80']),
    RTCIceServer(
        urls=[
            'turn:standard.relay.metered.ca:80',
            'turn:standard.relay.metered.ca:80?transport=tcp',
            'turn:standard.relay.metered.ca:443',
            'turn:standard.relay.metered.ca:443?transport=tcp',
        ],
        username   = METERED_USERNAME,
        credential = METERED_CREDENTIAL,
    ),
])


# ── AI audio track ────────────────────────────────────────────────────────────
class AIAudioTrack(MediaStreamTrack):
    kind = 'audio'

    def __init__(self):
        super().__init__()
        self._queue      = asyncio.Queue()
        self._sr         = 48000
        self._pts        = 0
        self._buf        = np.array([], dtype=np.int16)
        self._start_time = None
        self.is_playing  = False          # tracks if AI audio is actively playing

    def push(self, pcm_f32, sr):
        if len(pcm_f32) == 0:
            return
        if sr != self._sr:
            n       = int(len(pcm_f32) * self._sr / sr)
            pcm_f32 = np.interp(
                np.linspace(0, len(pcm_f32)-1, n),
                np.arange(len(pcm_f32)), pcm_f32
            ).astype(np.float32)
        pcm_f32         = np.clip(pcm_f32 * 1.5, -1.0, 1.0)
        pcm16           = (pcm_f32 * 32767).astype(np.int16)
        CHUNK           = 4800
        self.is_playing = True            # mark playing when audio queued
        for i in range(0, len(pcm16), CHUNK):
            self._queue.put_nowait(pcm16[i:i+CHUNK])
        print(f'[AITrack] Queued {len(pcm16)/self._sr:.2f}s of audio')

    async def recv(self):
        SAMPLES = 960
        loop    = asyncio.get_event_loop()

        if self._start_time is None:
            self._start_time = loop.time()

        target = self._start_time + (self._pts / self._sr)
        delta  = target - loop.time()
        if delta > 0:
            await asyncio.sleep(delta)

        while len(self._buf) < SAMPLES:
            try:
                chunk     = self._queue.get_nowait()
                self._buf = np.concatenate([self._buf, chunk])
            except asyncio.QueueEmpty:
                pad       = np.zeros(SAMPLES - len(self._buf), dtype=np.int16)
                self._buf = np.concatenate([self._buf, pad])
                break

        frame_data = self._buf[:SAMPLES]
        self._buf  = self._buf[SAMPLES:]

        # Mark playing=False when queue fully drained
        if self._queue.empty() and len(self._buf) == 0:
            if self.is_playing:
                print('[AITrack] Playback complete — mic listening resumed')
            self.is_playing = False

        frame             = av.AudioFrame(format='s16', layout='mono', samples=SAMPLES)
        frame.sample_rate = self._sr
        frame.pts         = self._pts
        frame.time_base   = fractions.Fraction(1, self._sr)
        frame.planes[0].update(frame_data.tobytes())
        self._pts += SAMPLES
        return frame


# ── Pipeline ──────────────────────────────────────────────────────────────────
async def run_pipeline(audio_np, sr, history, ai_track):
    print(f'[Pipeline] Transcribing {len(audio_np)/sr:.1f}s of audio...')
    text = transcribe(audio_np, sr)
    if not text:
        print('[Pipeline] Empty transcription — skipping')
        return
    print(f'[Pipeline] User said: "{text}"')
    reply   = ask_ai(text, history)
    emotion = detect_emotion(text + ' ' + reply)
    sr_out, audio_out = speak(reply, emotion)
    history.append((text, reply))
    if len(history) > 8:
        history.pop(0)
    print(f'[Pipeline] Pushing {len(audio_out)/sr_out:.2f}s audio to track...')
    ai_track.push(audio_out, sr_out)
    print(f'[Pipeline] ✅ Done')


async def _run_locked(audio_np, sr, history, ai_track, lock, pending):
    async with lock:
        await run_pipeline(audio_np, sr, history, ai_track)
    if pending[0] is not None:
        nxt, nsr   = pending[0]
        pending[0] = None
        asyncio.ensure_future(
            _run_locked(nxt, nsr, history, ai_track, lock, pending))


# ── FastAPI ───────────────────────────────────────────────────────────────────
app = FastAPI()

@app.get('/')
async def index():
    return HTMLResponse(HTML_TEMPLATE)


@app.post('/offer')
async def offer(request: Request):
    params   = await request.json()
    offer_sd = RTCSessionDescription(sdp=params['sdp'], type=params['type'])

    pc       = RTCPeerConnection(configuration=ICE_CONFIG)
    history  = []
    ai_track = AIAudioTrack()
    lock     = asyncio.Lock()
    pending  = [None]
    pcs.add(pc)

    pc.addTrack(ai_track)
    print(f'[Offer] AI track added, id={ai_track.id[:8]}')

    FRAME_DUR         = 0.02
    SILENCE_NEEDED    = int(SILENCE_THRESHOLD_SEC / FRAME_DUR)  # 100 frames = 2s
    PRE_SPEECH_FRAMES = int(PRE_SPEECH_KEEP_SEC   / FRAME_DUR)  # 15 frames = 0.3s

    audio_buf      = []
    silence_frames = [0]
    has_speech     = [False]
    speech_frames  = [0]
    frame_count    = [0]

    @pc.on('track')
    def on_track(track):
        print(f'[Track] Received kind={track.kind} id={track.id[:8]}')
        if track.kind != 'audio':
            return

        async def receive_audio():
            print('[Receive] Audio receive loop started ✅')
            while True:
                try:
                    frame = await asyncio.wait_for(track.recv(), timeout=15.0)
                except asyncio.TimeoutError:
                    print('[Receive] 15s timeout — no audio frames arriving')
                    continue
                except Exception as e:
                    print(f'[Receive] Loop ended: {e}')
                    break

                frame_count[0] += 1

                try:
                    arr = frame.to_ndarray()
                except Exception as e:
                    print(f'[Receive] Frame decode error: {e}')
                    continue

                if arr.ndim > 1:
                    arr = arr.mean(axis=0)
                arr = arr.astype(np.float32)

                if float(np.abs(arr).max()) > 2.0:
                    arr = arr / 32768.0

                sr  = frame.sample_rate
                rms = float(np.sqrt(np.mean(arr**2)))

                if frame_count[0] % 100 == 0:
                    print(f'[Receive] frame={frame_count[0]} sr={sr} '
                          f'rms={rms:.5f} shape={arr.shape} '
                          f'has_speech={has_speech[0]} '
                          f'speech_f={speech_frames[0]} '
                          f'silence_f={silence_frames[0]} '
                          f'pipeline_busy={lock.locked()} '
                          f'ai_playing={ai_track.is_playing}')

                is_speech = rms > ENERGY_THRESHOLD

                # ── Echo gate ─────────────────────────────────────────────────
                # Block mic while pipeline is computing OR AI audio is playing.
                # This prevents AI speaker output from leaking into mic and
                # being transcribed as a fake user utterance (peak=2.000 clips).
                if lock.locked() or ai_track.is_playing:
                    audio_buf.clear()
                    silence_frames[0] = 0
                    has_speech[0]     = False
                    speech_frames[0]  = 0
                    continue

                if is_speech:
                    audio_buf.append(arr)
                    silence_frames[0]  = 0
                    has_speech[0]      = True
                    speech_frames[0]  += 1

                elif has_speech[0]:
                    audio_buf.append(arr)
                    silence_frames[0] += 1

                    # Reset if silence accumulates too long with weak speech
                    # Prevents silence_f=300+ junk buildup that confuses Whisper
                    if (silence_frames[0] > SILENCE_NEEDED * 3
                            and speech_frames[0] < MIN_SPEECH_FRAMES):
                        print('[VAD] Long silence + weak speech — resetting buffer')
                        audio_buf.clear()
                        silence_frames[0] = 0
                        has_speech[0]     = False
                        speech_frames[0]  = 0

                else:
                    # Pre-speech rolling context window
                    audio_buf.append(arr)
                    if len(audio_buf) > PRE_SPEECH_FRAMES:
                        audio_buf.pop(0)

                buffered_sec = sum(len(a) for a in audio_buf) / max(sr, 1)

                do_process = (
                    has_speech[0]
                    and silence_frames[0] >= SILENCE_NEEDED
                    and speech_frames[0]  >= MIN_SPEECH_FRAMES
                    and not lock.locked()
                    and not ai_track.is_playing
                ) or (
                    buffered_sec > MAX_BUFFER_SEC
                    and not lock.locked()
                    and not ai_track.is_playing
                    and speech_frames[0] >= MIN_SPEECH_FRAMES
                )

                if do_process:
                    if buffered_sec >= MIN_SPEECH_SEC:
                        full = np.concatenate(audio_buf)
                        audio_buf.clear()
                        silence_frames[0] = 0
                        has_speech[0]     = False
                        speech_frames[0]  = 0
                        peak = float(np.abs(full).max())
                        print(f'[VAD] Triggered — {len(full)/sr:.2f}s clip '
                              f'peak={peak:.3f} '
                              f'rms={float(np.sqrt(np.mean(full**2))):.4f}')
                        asyncio.ensure_future(
                            _run_locked(full, sr, history,
                                        ai_track, lock, pending))
                    else:
                        print(f'[VAD] Clip too short ({buffered_sec:.2f}s) — discarded')
                        audio_buf.clear()
                        silence_frames[0] = 0
                        has_speech[0]     = False
                        speech_frames[0]  = 0

        asyncio.ensure_future(receive_audio())

    @pc.on('connectionstatechange')
    async def on_state():
        print(f'[WebRTC] Connection state → {pc.connectionState}')
        if pc.connectionState in ('failed', 'closed'):
            await pc.close()
            pcs.discard(pc)

    @pc.on('icegatheringstatechange')
    async def on_ice_gather():
        print(f'[ICE] Gathering state → {pc.iceGatheringState}')

    await pc.setRemoteDescription(offer_sd)
    answer = await pc.createAnswer()
    await pc.setLocalDescription(answer)

    print('[ICE] Waiting for server-side ICE gathering to complete...')
    if pc.iceGatheringState != 'complete':
        gather_done = asyncio.Event()

        @pc.on('icegatheringstatechange')
        async def _on_gather():
            if pc.iceGatheringState == 'complete':
                gather_done.set()

        try:
            await asyncio.wait_for(gather_done.wait(), timeout=10.0)
            print('[ICE] Server gathering complete ✅')
        except asyncio.TimeoutError:
            print('[ICE] Gathering timeout — sending with available candidates')

    print(f'[Offer] Returning answer — senders={len(pc.getSenders())} '
          f'iceState={pc.iceGatheringState}')

    return JSONResponse({
        'sdp' : pc.localDescription.sdp,
        'type': pc.localDescription.type,
    })

print('✅ WebRTC pipeline + FastAPI ready!')

✅ WebRTC pipeline + FastAPI ready!


In [7]:
HTML_TEMPLATE = """<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>Voice AI Call</title>
<style>
  *{box-sizing:border-box;margin:0;padding:0}
  body{
    font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;
    background:#0f0f1a;color:#e8e8f0;min-height:100vh;
    display:flex;align-items:center;justify-content:center;padding:20px;
  }
  .card{
    background:#1a1a2e;border:1px solid #2a2a4a;border-radius:24px;
    padding:40px 32px;width:100%;max-width:440px;text-align:center;
    box-shadow:0 20px 60px rgba(0,0,0,0.5);
  }
  .avatar{
    width:110px;height:110px;border-radius:50%;
    background:linear-gradient(135deg,#667eea,#764ba2);
    display:flex;align-items:center;justify-content:center;
    font-size:48px;margin:0 auto 20px;transition:box-shadow 0.3s;
  }
  .avatar.pulse{animation:pulse 1.4s ease-in-out infinite;}
  @keyframes pulse{
    0%,100%{box-shadow:0 0 0 8px rgba(102,126,234,0.3),0 0 0 16px rgba(102,126,234,0.1);}
    50%     {box-shadow:0 0 0 14px rgba(102,126,234,0.45),0 0 0 28px rgba(102,126,234,0.2);}
  }
  h1{font-size:22px;font-weight:700;margin-bottom:6px;}
  .sub{color:#666;font-size:12px;margin-bottom:24px;}
  .pill{
    display:inline-flex;align-items:center;gap:8px;
    padding:8px 20px;border-radius:999px;
    font-size:13px;font-weight:600;margin-bottom:20px;
    transition:all 0.3s;
  }
  .idle    {background:#1e1e3a;color:#555;   border:1px solid #2a2a4a;}
  .ready   {background:#0d2b1a;color:#34d399;border:1px solid #065f46;}
  .thinking{background:#2b1a00;color:#fbbf24;border:1px solid #92400e;}
  .error   {background:#2b0000;color:#f87171;border:1px solid #991b1b;}
  .dot{
    width:8px;height:8px;border-radius:50%;
    background:currentColor;animation:blink 1.2s ease-in-out infinite;
  }
  @keyframes blink{0%,100%{opacity:1}50%{opacity:0.15}}
  .call-btn{
    width:84px;height:84px;border-radius:50%;
    border:none;cursor:pointer;font-size:36px;
    display:flex;align-items:center;justify-content:center;
    margin:0 auto 20px;transition:all 0.2s;
  }
  .go  {background:#22c55e;box-shadow:0 6px 24px rgba(34,197,94,0.45);}
  .stop{background:#ef4444;box-shadow:0 6px 24px rgba(239,68,68,0.45);}
  .call-btn:hover {transform:scale(1.08);}
  .call-btn:active{transform:scale(0.95);}
  .vol-wrap{margin-bottom:20px;display:none;}
  .vol-wrap.on{display:block;}
  .vol-label{font-size:11px;color:#444;margin-bottom:4px;}
  .vol-bar{height:6px;border-radius:3px;background:#1e1e3a;overflow:hidden;}
  .vol-fill{
    height:100%;border-radius:3px;
    background:linear-gradient(90deg,#667eea,#34d399);
    width:0%;transition:width 0.08s;
  }
  .transcript-box{
    background:#080815;border:1px solid #1e1e3a;border-radius:14px;
    padding:14px 12px;text-align:left;max-height:280px;overflow-y:auto;
    font-size:13px;margin-top:16px;display:none;line-height:1.6;
  }
  .transcript-box.on{display:block;}
  .bubble{margin-bottom:12px;padding:8px 12px;border-radius:12px;max-width:90%;}
  .bubble.you{background:#1e1e4a;color:#818cf8;margin-left:auto;text-align:right;}
  .bubble.ai {background:#0d2b1a;color:#34d399;}
  .bubble .label{font-size:10px;opacity:0.6;margin-bottom:2px;}
  .log-box{
    background:#080815;border:1px solid #1e1e3a;border-radius:14px;
    padding:10px 12px;text-align:left;max-height:120px;overflow-y:auto;
    font-size:11px;font-family:'Courier New',monospace;
    margin-top:8px;display:none;
  }
  .log-box.on{display:block;}
  .log-box .sys{color:#3a3a5a;}
  .log-box .err{color:#f87171;}
  details{margin-top:8px;cursor:pointer;font-size:11px;color:#444;}
</style>
</head>
<body>
<div class="card">
  <div class="avatar" id="avatar">🤖</div>
  <h1>Voice AI</h1>
  <p class="sub">WebRTC · Whisper · Groq · Kokoro</p>

  <div class="pill idle" id="pill">
    <span class="dot"></span>
    <span id="ptxt">Press 📞 to start</span>
  </div>

  <div class="vol-wrap" id="volWrap">
    <div class="vol-label">Your mic level</div>
    <div class="vol-bar"><div class="vol-fill" id="volFill"></div></div>
  </div>

  <button class="call-btn go" id="btn" onclick="toggleCall()">📞</button>

  <!-- Conversation transcript -->
  <div class="transcript-box" id="transcript"></div>

  <!-- Debug log (collapsed) -->
  <details>
    <summary>Debug log</summary>
    <div class="log-box" id="log"></div>
  </details>
</div>

<audio id="aiAudio" autoplay playsinline></audio>

<script>
const METERED_USERNAME   = '""" + METERED_USERNAME + """';
const METERED_CREDENTIAL = '""" + METERED_CREDENTIAL + """';

const ICE_CONFIG = {
  iceServers: [
    {urls: 'stun:stun.metered.ca:80'},
    {
      urls: [
        'turn:standard.relay.metered.ca:80',
        'turn:standard.relay.metered.ca:80?transport=tcp',
        'turn:standard.relay.metered.ca:443',
        'turn:standard.relay.metered.ca:443?transport=tcp',
      ],
      username  : METERED_USERNAME,
      credential: METERED_CREDENTIAL,
    },
  ],
  iceTransportPolicy  : 'relay',
  iceCandidatePoolSize: 10,
};

let pc=null, localStream=null, inCall=false;
let audioCtx=null, volRaf=null;

// ── Logging ────────────────────────────────────────────────────────────────
function addLog(msg, cls='sys') {
  const el = document.getElementById('log');
  el.classList.add('on');
  const d  = document.createElement('div');
  d.className = cls;
  d.textContent = new Date().toLocaleTimeString('en',{hour12:false}) + '  ' + msg;
  el.appendChild(d);
  el.scrollTop = el.scrollHeight;
}

function addTranscript(speaker, text) {
  const box = document.getElementById('transcript');
  box.classList.add('on');
  const b   = document.createElement('div');
  b.className = 'bubble ' + (speaker === 'You' ? 'you' : 'ai');
  b.innerHTML = `<div class="label">${speaker}</div><div>${text}</div>`;
  box.appendChild(b);
  box.scrollTop = box.scrollHeight;
}

function setStatus(cls, txt) {
  document.getElementById('pill').className = 'pill ' + cls;
  document.getElementById('ptxt').textContent = txt;
}

function setAIPulse(on) {
  document.getElementById('avatar').classList.toggle('pulse', on);
}

// ── Mic volume meter ───────────────────────────────────────────────────────
function startVolMeter(stream) {
  audioCtx    = new (window.AudioContext || window.webkitAudioContext)();
  const src   = audioCtx.createMediaStreamSource(stream);
  const anal  = audioCtx.createAnalyser();
  anal.fftSize = 512;
  src.connect(anal);
  document.getElementById('volWrap').classList.add('on');
  const data  = new Uint8Array(anal.frequencyBinCount);
  function tick() {
    anal.getByteFrequencyData(data);
    const avg = data.reduce((a,b)=>a+b,0)/data.length;
    document.getElementById('volFill').style.width = Math.min(avg*3.0,100)+'%';
    volRaf = requestAnimationFrame(tick);
  }
  tick();
}

function stopVolMeter() {
  if (volRaf) cancelAnimationFrame(volRaf);
  document.getElementById('volWrap').classList.remove('on');
  if (audioCtx) { audioCtx.close(); audioCtx=null; }
}

// ── Watch AI audio output and pulse avatar ─────────────────────────────────
let aiWatchCtx = null;
function watchAIAudio(stream) {
  if (aiWatchCtx) { aiWatchCtx.close(); aiWatchCtx=null; }
  aiWatchCtx     = new (window.AudioContext||window.webkitAudioContext)();
  const src      = aiWatchCtx.createMediaStreamSource(stream);
  const anal     = aiWatchCtx.createAnalyser();
  anal.fftSize   = 256;
  src.connect(anal);
  const data     = new Uint8Array(anal.frequencyBinCount);
  let wasSpeaking = false;
  setInterval(()=>{
    anal.getByteFrequencyData(data);
    const avg = data.reduce((a,b)=>a+b,0)/data.length;
    const speaking = avg > 5;
    setAIPulse(speaking);
    if (speaking && !wasSpeaking) {
      setStatus('thinking','🔊 AI speaking...');
    } else if (!speaking && wasSpeaking && inCall) {
      setStatus('ready','✅ Connected — speak now');
    }
    wasSpeaking = speaking;
  }, 80);
}

// ── Start call ─────────────────────────────────────────────────────────────
async function startCall() {
  setStatus('idle','Requesting mic...');
  addLog('Requesting microphone...');

  try {
    localStream = await navigator.mediaDevices.getUserMedia({
      audio:{
        echoCancellation:true,
        noiseSuppression:true,
        autoGainControl :true,
        channelCount    :1,
        sampleRate      :48000,
      },
      video:false
    });
  } catch(e) {
    setStatus('error','Mic denied — allow mic and reload');
    addLog('Mic error: '+e.message, 'err');
    return;
  }
  addLog('Mic OK — tracks: '+localStream.getAudioTracks().length);
  startVolMeter(localStream);

  pc = new RTCPeerConnection(ICE_CONFIG);

  localStream.getTracks().forEach(t => {
    pc.addTrack(t, localStream);
    addLog('Added mic track: '+t.kind+' '+t.label.slice(0,30));
  });

  // ── Receive AI audio ──────────────────────────────────────────────────────
  pc.ontrack = (e) => {
    addLog('Remote track received: '+e.track.kind+' readyState='+e.track.readyState);
    const audio = document.getElementById('aiAudio');

    // Use stream directly if available, else wrap in new MediaStream
    const stream = (e.streams && e.streams[0])
        ? e.streams[0]
        : new MediaStream([e.track]);

    audio.srcObject = stream;

    // Must resume AudioContext after user gesture (browser autoplay policy)
    audio.play().then(()=>{
      addLog('AI audio playback started ✅');
      watchAIAudio(stream);
    }).catch(err => {
      addLog('Audio autoplay blocked: '+err+' — click anywhere to resume', 'err');
      // Fallback: resume on next user click
      document.addEventListener('click', ()=>{
        audio.play().then(()=>{ addLog('Audio resumed ✅'); watchAIAudio(stream); });
      }, {once:true});
    });
  };

  pc.onconnectionstatechange = () => {
    const s = pc.connectionState;
    addLog('WebRTC state: '+s, s==='failed'||s==='disconnected' ? 'err' : 'sys');
    if (s==='connected')    setStatus('ready','✅ Connected — speak now');
    if (s==='connecting')   setStatus('idle','Connecting...');
    if (s==='disconnected') setStatus('error','Disconnected');
    if (s==='failed')       { setStatus('error','Connection failed'); endCall(); }
  };

  pc.onicegatheringstatechange  = () => addLog('ICE gather: '+pc.iceGatheringState);
  pc.oniceconnectionstatechange = () => addLog('ICE conn:   '+pc.iceConnectionState);
  pc.onicecandidate = (e) => {
    if (e.candidate) addLog('ICE candidate: '+e.candidate.type);
  };

  setStatus('idle','Creating SDP offer...');
  const offer = await pc.createOffer({offerToReceiveAudio:true});
  await pc.setLocalDescription(offer);
  addLog('Offer created, waiting for ICE...');

  // Wait for browser ICE gathering (max 6 s)
  await new Promise(resolve => {
    if (pc.iceGatheringState==='complete') { resolve(); return; }
    const t = setTimeout(resolve, 6000);
    pc.addEventListener('icegatheringstatechange', () => {
      if (pc.iceGatheringState==='complete') { clearTimeout(t); resolve(); }
    });
  });
  addLog('ICE complete — sending offer to server');

  // Send to Python server
  let resp;
  try {
    resp = await fetch('/offer', {
      method :'POST',
      headers:{'Content-Type':'application/json'},
      body   : JSON.stringify({
        sdp : pc.localDescription.sdp,
        type: pc.localDescription.type,
      })
    });
  } catch(e) {
    setStatus('error','Server unreachable');
    addLog('Fetch error: '+e, 'err'); return;
  }

  if (!resp.ok) {
    setStatus('error','Server error '+resp.status);
    addLog('Server HTTP error: '+resp.status, 'err'); return;
  }

  const answer = await resp.json();
  addLog('Answer received — applying remote description');
  await pc.setRemoteDescription(new RTCSessionDescription(answer));

  inCall = true;
  document.getElementById('btn').className   = 'call-btn stop';
  document.getElementById('btn').textContent = '📵';
  setStatus('ready','✅ Connected — speak now');
  addLog('✅ Call live! Speak now.');
}

// ── End call ───────────────────────────────────────────────────────────────
function endCall() {
  if (pc)          { pc.close(); pc=null; }
  if (localStream) { localStream.getTracks().forEach(t=>t.stop()); localStream=null; }
  if (aiWatchCtx)  { aiWatchCtx.close(); aiWatchCtx=null; }
  stopVolMeter();
  setAIPulse(false);
  inCall = false;
  document.getElementById('btn').className   = 'call-btn go';
  document.getElementById('btn').textContent = '📞';
  document.getElementById('aiAudio').srcObject = null;
  setStatus('idle','Call ended — press 📞 to call again');
  addLog('Call ended.');
}

function toggleCall() { if (inCall) endCall(); else startCall(); }
</script>
</body>
</html>"""

print('✅ HTML template ready!')

✅ HTML template ready!


In [8]:
import uvicorn, threading, subprocess, time
from pyngrok import ngrok, conf

# ── Kill anything on port 8000 ────────────────────────────────────────────────
try:
    subprocess.run(['fuser', '-k', '8000/tcp'], capture_output=True, timeout=3)
    print('🔪 Cleared port 8000')
except Exception:
    pass
time.sleep(1)

# ── Test TURN credentials before starting ─────────────────────────────────────
print('🔍 Testing TURN relay...')
turn_result = subprocess.run(
    ['python3', '-c', f"""
import asyncio
from aiortc import RTCPeerConnection, RTCConfiguration, RTCIceServer

async def test():
    cfg = RTCConfiguration(iceServers=[
        RTCIceServer(
            urls=['turn:standard.relay.metered.ca:80?transport=tcp'],
            username='{METERED_USERNAME}',
            credential='{METERED_CREDENTIAL}',
        )
    ])
    pc = RTCPeerConnection(configuration=cfg)
    await pc.setLocalDescription(await pc.createOffer(offerToReceiveAudio=True))
    await asyncio.sleep(5)
    sdp = pc.localDescription.sdp
    relays = [l for l in sdp.split('\\\\n') if 'relay' in l.lower()]
    print('RELAY candidates:', len(relays))
    print('✅ TURN working!' if relays else '❌ TURN failed — check credentials')
    await pc.close()

asyncio.run(test())
"""],
    capture_output=True, text=True, timeout=20
)
print(turn_result.stdout.strip())
if '❌' in turn_result.stdout:
    print('⚠️  TURN test failed — calls may not connect.')
    print('    Get credentials at: https://dashboard.metered.ca')

# ── ngrok ─────────────────────────────────────────────────────────────────────
ngrok.kill()
conf.get_default().auth_token = NGROK_TOKEN
tunnel  = ngrok.connect(8000, 'http')
pub_url = tunnel.public_url.replace('http://', 'https://')

print()
print('='*58)
print('  WebRTC VOICE AI CALL — READY')
print('='*58)
print(f'  Open in browser : {pub_url}')
print(f'  WebRTC offer    : {pub_url}/offer')
print('='*58)
print('  Works on phone, tablet, laptop!')
print('  Press 📞 to start call.')
print('  Interrupt kernel to stop.')
print('='*58)

# ── Launch uvicorn in background thread (fixes Colab event loop conflict) ──────
config = uvicorn.Config(app, host='0.0.0.0', port=8000, log_level='warning')
server = uvicorn.Server(config)
thread = threading.Thread(target=server.run, daemon=True)
thread.start()
thread.join()

🔪 Cleared port 8000
🔍 Testing TURN relay...


  WebRTC VOICE AI CALL — READY
  Open in browser : https://poshly-climant-natalya.ngrok-free.dev
  WebRTC offer    : https://poshly-climant-natalya.ngrok-free.dev/offer
  Works on phone, tablet, laptop!
  Press 📞 to start call.
  Interrupt kernel to stop.
[Offer] AI track added, id=9b2f7143
[Track] Received kind=audio id=1c51fb65
[Receive] Audio receive loop started ✅
[ICE] Gathering state → gathering
[ICE] Gathering state → complete
[ICE] Waiting for server-side ICE gathering to complete...
[Offer] Returning answer — senders=1 iceState=complete
[WebRTC] Connection state → connecting
[WebRTC] Connection state → connected
[Receive] frame=100 sr=48000 rms=0.12354 shape=(1920,) has_speech=True speech_f=10 silence_f=0 pipeline_busy=False ai_playing=False
[Receive] frame=200 sr=48000 rms=0.04691 shape=(1920,) has_speech=True speech_f=108 silence_f=0 pipeline_busy=False ai_playing=False
[Receive] frame=300 sr=48000 rms=0.00125 shape=(1920,) has_s

ERROR:asyncio:Exception in callback <bound method Transaction.__retry of <aioice.stun.Transaction object at 0x7ac74bf4b230>>
handle: <TimerHandle Transaction.__retry>
Traceback (most recent call last):
  File "uvloop/cbhandles.pyx", line 253, in uvloop.loop.TimerHandle._run
  File "/usr/local/lib/python3.12/dist-packages/aioice/stun.py", line 330, in __retry
    self.__future.set_exception(TransactionTimeout())
asyncio.exceptions.InvalidStateError: invalid state


   AI: "You must have been pretty busy then, anything specific that kept you occupied?"
   TTS generated 5.03s audio, max=0.371
[Pipeline] Pushing 5.03s audio to track...
[AITrack] Queued 5.03s of audio
[Pipeline] ✅ Done
[Receive] frame=3600 sr=48000 rms=0.00142 shape=(1920,) has_speech=False speech_f=0 silence_f=0 pipeline_busy=False ai_playing=True
[Receive] frame=3700 sr=48000 rms=0.00113 shape=(1920,) has_speech=False speech_f=0 silence_f=0 pipeline_busy=False ai_playing=True
[AITrack] Playback complete — mic listening resumed
[Receive] frame=3800 sr=48000 rms=0.00122 shape=(1920,) has_speech=False speech_f=0 silence_f=0 pipeline_busy=False ai_playing=False
[Receive] frame=3900 sr=48000 rms=0.04830 shape=(1920,) has_speech=True speech_f=34 silence_f=0 pipeline_busy=False ai_playing=False
[Receive] frame=4000 sr=48000 rms=0.00101 shape=(1920,) has_speech=True speech_f=103 silence_f=21 pipeline_busy=False ai_playing=False
[VAD] Triggered — 9.44s clip peak=0.176 rms=0.0267
[Pipeline] 

KeyboardInterrupt: 